In [9]:
import pandas as pd
import numpy as np

import networkx as nx

import matplotlib.pyplot as plt

from scipy.stats import linregress, gaussian_kde

import powerlaw

import re
import os
import glob

# # Supress warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

def get_network(df, dataset, network_type, net_id):

    nodes = set(df.index.astype(str)).union(set(df.columns.astype(str)))

    # Create graph
    G = nx.Graph()

    for node in nodes:
        G.add_node(node)
        
    for i, row in df.iterrows():
        for j, val in row.items():
            if val != 0:
                G.add_edge(str(i), str(j))

    if nx.is_bipartite(G):
        try:
            bottom_nodes, top_nodes = nx.bipartite.sets(G)
            nx.set_node_attributes(
                G,
                {n: "bottom" for n in bottom_nodes},
                name="bipartite_set",
            )
            nx.set_node_attributes(
                G,
                {n: "top" for n in top_nodes},
                name="bipartite_set",
            )
        except Exception:
            
            bottom_nodes = set(df.index.astype(str))
            top_nodes = set(df.columns.astype(str))
            
            nx.set_node_attributes(
                G,
                {n: "bottom" for n in bottom_nodes},
                name="bipartite_set",
            )
            nx.set_node_attributes(
                G,
                {n: "top" for n in top_nodes},
                name="bipartite_set",
            )
            
    # Graph metadata
    G.graph["dataset"] = dataset
    G.graph["network_type"] = network_type
    G.graph["ID"] = net_id

    nx.write_graphml(G, f"Data/networks/{net_id}.graphml")

def generate_wol_networks():

    web_of_life_translation = {
        "FW": "food_web",
        "AHP": "host_parasite",
        "MPL": "plant_pollinator",
        "MSD": "seed_dispersal",
        "MPA": "plant_ant",
        "MAF": "anemone_fish",
    }
    
    for folder in os.listdir("Data/web_of_life"):
        
        print(f"Processing folder: {folder}")
        
        filenames = glob.glob(f"Data/web_of_life/{folder}/*.csv")

        # remove references.csv if present
        filenames = [f for f in filenames if not f.endswith("references.csv")]

        for filename in filenames:

            print(f"\tProcessing file: {filename}")

            id = filename.split("/")[-1].split(".")[0]
            dataset = "web_of_life"
            network_type = web_of_life_translation.get(
                re.sub(r"[_\d]", "", id), np.nan
            )

            df = pd.read_csv(
                filename, index_col=0
            ) 

            if df.iloc[0, 0] == "#":
                df = df.iloc[1:, 1:]

            get_network(
                df,
                dataset,
                network_type,
                id,
            )

def giant_component_graph(G):
    # Work with undirected simple graph for these structural metrics
    H = G if not G.is_directed() else G.to_undirected()
    if H.number_of_nodes() == 0:
        return H
    if nx.is_connected(H):
        return H
    gcc = max(nx.connected_components(H), key=len)
    return H.subgraph(gcc).copy()

def network_based_metrics(filenames):

    out_dir = "Results/Communities_A/"
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)

    # --- Output columns: keep yours + add new ones ---
    df_results = pd.DataFrame(
        columns=[
            "dataset",
            "network_type",
            "id",
            "S",
            "L",
            "C",
            "density",
            "diffusion_time",
            "clustering_coefficient",
            "transitivity",
            "average_shortest_path",
            "mean_degree",
            "var_degree",
            "modularity",
            "global_efficiency",
            "assortativity",
            "frac_2core",
            "is_bipartite",
        ]
    )

    for idx, filename in enumerate(filenames, start=1):

        print(f"\rProcessing file {idx}/{len(filenames)}", end="")

        G = nx.read_graphml(filename)

        dataset = G.graph.get("dataset", np.nan)
        network_type = G.graph.get("network_type", np.nan)
        id_ = G.graph.get("ID", np.nan)
        
        S = G.number_of_nodes()
        L = G.number_of_edges()

        # Density
        rho = nx.density(G)

        # Connectance
        is_bipartite = nx.is_bipartite(G)

        if is_bipartite:
            try:
                bottom_nodes, top_nodes = nx.bipartite.sets(G)
                C = nx.bipartite.density(G, bottom_nodes)
            except Exception:
                # Get bipartite sets from node attributes if available
                bottom_nodes_attr = {
                    n: d.get("bipartite_set") == "bottom" for n, d in G.nodes(data=True)
                }
                if any(bottom_nodes_attr.values()):
                    bottom_nodes = {n for n, is_bottom in bottom_nodes_attr.items() if is_bottom}
                    C = nx.bipartite.density(G, bottom_nodes)
                else:
                    C = np.nan
                
        else:
            C = rho

        # Degree stats
        degree_dist = np.array([d for _, d in G.degree()], dtype=float)
        df_degree = pd.DataFrame({"degree": degree_dist})

        mean_degree = float(np.mean(degree_dist))
        var_degree = float(np.var(degree_dist))

        # Giant connected component for path length + Laplacian spectrum
        G_gcc = giant_component_graph(G)

        # Clustering (mean local) + transitivity (global)
        try:
            clustering_coefficient = float(nx.average_clustering(G))
        except Exception:
            clustering_coefficient = np.nan

        try:
            transitivity = float(nx.transitivity(G))
        except Exception:
            transitivity = np.nan

        # Average shortest path (use GCC to avoid disconnected errors)
        try:
            average_shortest_path = float(nx.average_shortest_path_length(G_gcc))
        except Exception:
            average_shortest_path = np.nan

        # Diffusion time (1/lambda2 of Laplacian on GCC)
        try:
            Ls = np.sort(nx.laplacian_spectrum(G_gcc))
            Ls = Ls[Ls > 1e-12]
            t_diff = float(1.0 / Ls[0]) if len(Ls) else np.nan
        except Exception:
            t_diff = np.nan

        # Modularity + number of communities (use one partition consistently)
        try:
            comms = nx.community.greedy_modularity_communities(G)
            modularity = float(nx.community.modularity(G, comms))
        except Exception:
            modularity = np.nan

        # Global efficiency (can be slow for large graphs; keep try/except)
        try:
            global_efficiency = float(nx.global_efficiency(G))
        except Exception:
            global_efficiency = np.nan

        # Assortativity
        try:
            assortativity = float(nx.degree_assortativity_coefficient(G))
        except Exception:
            assortativity = np.nan

        # k-core / core-periphery proxies (simple and robust)
        try:
            frac_2core = (
                float(nx.k_core(G, k=2).number_of_nodes() / G.number_of_nodes())
                if G.number_of_nodes()
                else np.nan
            )
        except Exception:
            frac_2core = np.nan

        # Assemble row
        df_i = pd.DataFrame(
            {
                "dataset": [dataset],
                "network_type": [network_type],
                "id": [id_],
                "S": [S],
                "L": [L],
                "C": [C],
                "density": [rho],
                "diffusion_time": [t_diff],
                "clustering_coefficient": [clustering_coefficient],
                "transitivity": [transitivity],
                "average_shortest_path": [average_shortest_path],
                "mean_degree": [mean_degree],
                "var_degree": [var_degree],
                "modularity": [modularity],
                "global_efficiency": [global_efficiency],
                "assortativity": [assortativity],
                "frac_2core": [frac_2core],
                "is_bipartite": [is_bipartite],
            }
        )

        df_results = pd.concat([df_results, df_i], ignore_index=True)

        df_degree.to_csv(
            os.path.join(out_dir, f"degree_distribution_{id_}.csv"),
            index=False,
        )

    df_results.to_csv("Results/field_studies_metrics_network.csv", index=False)

    return df_results

# Generate networks from Web of Life datasets

In [ ]:
# generate_wol_networks()

# Add network_type to mangal networks

In [ ]:
filenames = glob.glob("Data/networks/*.graphml")

df_metadata = pd.read_csv("Data/mangal/mangal_networks_metadata.csv")

for filename in filenames:

    if "mangal" in filename:

        G = nx.read_graphml(filename)

        if "network_type" not in G.graph:
            id_ = G.graph.get("ID", np.nan)
            metadata_row = df_metadata[df_metadata["id"] == id_]
            if not metadata_row.empty:
                G.graph["network_type"] = metadata_row["interaction_type"].values[0]
            else:
                G.graph["network_type"] = np.nan
                print(f"Warning: No metadata found for {id_} in {filename}")
                
        nx.write_graphml(G, filename)

# Generate network dataset

In [10]:
filenames = glob.glob("Data/networks/*.graphml")

df_networks_metrics = network_based_metrics(filenames)

Processing file 1801/1801

In [3]:
len(filenames)

1801